# Vision Transformer on CIFAR-10

这个 Notebook 从 `CIFAR-10` 数据集加载开始，完整展示 `Resize(224x224) -> Vision Transformer (ViT)` 的训练与结构分析流程。

内容包括：
- 数据集预处理与可视化
- `DataLoader` 构建
- Patch Embedding 实现
- Class Token、Position Embedding、Transformer Encoder 解读
- Token 尺寸变化分析
- 训练、验证与预测展示
- ViT 与经典 CNN 的核心差异说明

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# 数学工具用于可视化排版等辅助逻辑
import math
# dataclass 用于统一管理实验配置
from dataclasses import dataclass

# matplotlib 用于样本和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# torchvision 提供常见数据集与图像预处理工具
from torchvision import datasets, transforms

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，便于复现
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据下载与缓存目录
    data_root: str = './data'
    # 将 CIFAR-10 从 32x32 拉伸到 224x224
    image_size: int = 224
    # ViT 通常把图像切成固定大小的 patch
    patch_size: int = 16
    # 每个 batch 的样本数
    batch_size: int = 64
    # DataLoader 并行加载进程数
    num_workers: int = 2
    # 学习率
    lr: float = 1e-3
    # 训练轮数
    epochs: int = 5
    # 下面这些是教学版 ViT 的核心结构参数
    embed_dim: int = 256
    depth: int = 6
    num_heads: int = 8
    mlp_ratio: float = 4.0
    dropout: float = 0.1


cfg = Config()
cfg

## 2. 加载 CIFAR-10 并拉伸到 224x224

ViT 通常在较大分辨率输入上更容易说明 patch 切分逻辑。这里先把 `CIFAR-10` 拉伸到 `224x224`，再切成 `16x16` patch。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# 训练集：尺寸拉伸、随机翻转、张量化、归一化
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 测试集不做随机增强，只保留基础预处理
test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 本地没有数据时自动下载
train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=test_transform,
)

# 类别名称
classes = train_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于可视化显示
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 展示若干样本，确认输入图像形式
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), range(8)):
    image, label = train_dataset[idx]
    image = denormalize(image, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 samples resized to 224x224', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 训练集打乱顺序，减少模型对样本顺序的依赖
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 测试集保持固定顺序即可
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 检查一个 batch 的维度
images, labels = next(iter(train_loader))
print('batch image shape:', images.shape)
print('batch label shape:', labels.shape)

## 4. ViT 实现

为了方便讲清结构，这里实现一个教学清晰版的小型 ViT。它保留了 ViT 的关键思想：
- 把图像切成 patch
- 每个 patch 映射成 token 向量
- 加入 `class token` 和位置编码
- 用 Transformer Encoder 做全局建模
- 用 `class token` 完成分类

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=256):
        super().__init__()
        # 图像大小必须能被 patch 大小整除
        assert image_size % patch_size == 0
        self.image_size = image_size
        self.patch_size = patch_size
        # patch 数量 = (H / P) * (W / P)
        self.num_patches = (image_size // patch_size) ** 2
        # 用一个步幅等于 patch 大小的卷积同时完成切块和线性映射
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        # 输出形状先变成 [B, embed_dim, H/P, W/P]
        x = self.proj(x)
        # 展平成 patch 序列，再转成 [B, num_patches, embed_dim]
        x = x.flatten(2).transpose(1, 2)
        return x


class VisionTransformerCIFAR10(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        num_classes=10,
        embed_dim=256,
        depth=6,
        num_heads=8,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=3,
            embed_dim=embed_dim,
        )
        num_patches = self.patch_embed.num_patches

        # class token 是一个可学习向量，最终用于聚合全局分类信息
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # 位置编码帮助模型区分 patch 的空间位置
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        # Transformer Encoder 的单层配置
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        # 堆叠多层 encoder，完成全局 token 建模
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        # 用 class token 的最终表示做分类
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # 把图像切成 patch token 序列
        x = self.patch_embed(x)
        batch_size = x.size(0)

        # 为每个样本复制一个 class token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        # 在 patch token 前面拼接 class token
        x = torch.cat((cls_tokens, x), dim=1)
        # 加上可学习位置编码
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # Transformer Encoder 建模 token 之间的全局关系
        x = self.encoder(x)
        x = self.norm(x)

        # 取第一个 token，也就是 class token 的输出做分类
        cls_output = x[:, 0]
        logits = self.head(cls_output)
        return logits


model = VisionTransformerCIFAR10(
    image_size=cfg.image_size,
    patch_size=cfg.patch_size,
    num_classes=10,
    embed_dim=cfg.embed_dim,
    depth=cfg.depth,
    num_heads=cfg.num_heads,
    mlp_ratio=cfg.mlp_ratio,
    dropout=cfg.dropout,
).to(device)
model

## 5. Token 尺寸变化分析

与 CNN 不同，ViT 的关键不再是特征图尺寸逐层缩小，而是 token 序列如何形成、如何进入 Transformer。下面打印关键阶段的张量形状。

In [ ]:
def inspect_token_shapes(model, input_shape=(1, 3, 224, 224)):
    # 构造一个假的输入，只用于分析张量形状
    x = torch.randn(input_shape)
    print(f'input          -> {tuple(x.shape)}')

    x = model.patch_embed(x)
    print(f'patch_embed    -> {tuple(x.shape)}')

    batch_size = x.size(0)
    cls_tokens = model.cls_token.expand(batch_size, -1, -1)
    x = torch.cat((cls_tokens, x), dim=1)
    print(f'+ cls_token     -> {tuple(x.shape)}')

    x = x + model.pos_embed
    print(f'+ pos_embed     -> {tuple(x.shape)}')

    x = model.encoder(x)
    print(f'encoder output  -> {tuple(x.shape)}')

    x = model.norm(x)
    cls_output = x[:, 0]
    print(f'class token out -> {tuple(cls_output.shape)}')

    logits = model.head(cls_output)
    print(f'logits          -> {tuple(logits.shape)}')


inspect_token_shapes(model.cpu())
model = model.to(device)

## 6. ViT 的关键机制解读

1. Patch Embedding
   - 图像不再直接走卷积堆叠，而是先切成多个小块，每个 patch 对应一个 token。

2. Class Token
   - 类似一个专门负责汇总全局信息的占位符，最后用它的输出做分类。

3. Position Embedding
   - Transformer 本身不天然理解空间顺序，因此需要显式加入位置编码。

4. Self-Attention
   - 每个 token 都可以和所有其他 token 建立关系，这和 CNN 依赖局部卷积核的思路不同。

5. ViT 和 CNN 的差异
   - CNN 强调局部归纳偏置。
   - ViT 更强调全局关系建模，但通常对数据量和训练策略更敏感。

## 7. 参数量统计

In [ ]:
def count_parameters(model):
    # 只统计可训练参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total_params = count_parameters(model)
print(f'Trainable parameters: {total_params:,}')

## 8. 训练与验证函数

In [ ]:
# 多分类任务常用交叉熵损失
criterion = nn.CrossEntropyLoss()
# 使用 Adam 便于快速开始实验
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 训练模式
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 把一个 batch 的数据送到目标设备
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    # 评估模式
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

## 9. 训练主循环

ViT 对训练资源和训练策略通常比较敏感。这里默认只提供完整训练流程，不主动执行训练命令。

In [ ]:
# 记录训练和验证指标，便于后面画曲线
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

# 每轮先训练，再评估
for epoch in range(cfg.epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

In [ ]:
# 绘制损失曲线和准确率曲线
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='train loss')
axes[0].plot(epochs, history['val_loss'], label='val loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='train acc')
axes[1].plot(epochs, history['val_acc'], label='val acc')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, num_images=8):
    # 切换到评估模式，保证推理行为稳定
    model.eval()
    # 取一个 batch 做展示
    images, labels = next(iter(dataloader))
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    preds = logits.argmax(dim=1)

    fig, axes = plt.subplots(2, math.ceil(num_images / 2), figsize=(16, 6))
    axes = axes.flatten()

    for i in range(num_images):
        # 反归一化后再显示图像
        image = denormalize(images[i].cpu(), cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
        axes[i].imshow(image)
        axes[i].set_title(f'true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}')
        axes[i].axis('off')

    for i in range(num_images, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, classes, device)